In [9]:
from rdkit.Chem import Descriptors, MolFromSmiles
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score, matthews_corrcoef, f1_score
from pandas import DataFrame, concat
from pickle import load

In [10]:
#создаем словарь из дескприторов структуры
ConstDescriptors = {"HeavyAtomCount" : Descriptors.HeavyAtomCount,
                       "NHOHCount" : Descriptors.NHOHCount,
                       "NOCount" : Descriptors.NOCount,
                       "NumHAcceptors" : Descriptors.NumHAcceptors,
                       "NumHDonors" : Descriptors.NumHDonors,
                       "NumHeteroatoms" : Descriptors.NumHeteroatoms,
                       "NumRotatableBonds" : Descriptors.NumRotatableBonds,
                       "NumValenceElectrons" : Descriptors.NumValenceElectrons,
                       "NumAromaticRings" : Descriptors.NumAromaticRings,
                       "NumAliphaticHeterocycles" : Descriptors.NumAliphaticHeterocycles,
                       "RingCount" : Descriptors.RingCount}

#создаем словарь из физико-химических дескрипторов                            
PhisChemDescriptors = {"MW" : Descriptors.MolWt,
                          "LogP" : Descriptors.MolLogP,
                          "MR" : Descriptors.MolMR,
                          "TPSA" : Descriptors.TPSA}

#объединяем все дескрипторы в один словарь
descriptors = {}
descriptors.update(ConstDescriptors)
descriptors.update(PhisChemDescriptors)
print(len(descriptors), "количество дескрипторов в словаре")


#функция для генерации дескрипторов из молекул
def mol_dsc_calc(mols):
    
    return DataFrame({k: f(m) for k, f in descriptors.items()} 
             for m in mols)


with open('../data/classifier/modeling_data.pickle', 'rb') as file:
    data = load(file)
#оформляем sklearn трансформер для использования в конвеерном моделировании (sklearn Pipeline)
descriptors_transformer = FunctionTransformer(mol_dsc_calc, validate=False)

15 количество дескрипторов в словаре


In [11]:
molecules = [
    MolFromSmiles(mol) for mol in data['SMILES']
]

In [12]:
X = descriptors_transformer.transform(molecules)
Y = data['Activity']
XY = concat((X, Y), axis=1)
XY

,HeavyAtomCount,NHOHCount,NOCount,NumHAcceptors,NumHDonors,NumHeteroatoms,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumAliphaticHeterocycles,RingCount,MW,LogP,MR,TPSA,Activity
0,32,0,8,7,0,8,5,164,3,2,5,434.448,3.48760,113.2495,91.10,1
1,23,0,5,4,0,6,3,122,1,2,4,319.332,2.05930,78.1170,55.84,1
2,23,1,5,4,1,6,2,126,1,2,3,323.364,2.73460,81.6418,59.00,0
3,24,1,4,3,1,5,3,124,2,2,4,327.355,2.32120,85.9918,49.77,1
4,17,3,3,2,3,3,0,88,2,1,4,228.295,1.53040,67.1582,48.05,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75924,24,1,5,4,1,6,4,122,3,0,4,340.814,3.42850,94.0122,59.81,1
75925,28,1,6,5,1,6,7,140,4,0,4,373.412,3.80440,106.2408,77.24,1
75926,32,3,9,7,2,9,7,168,3,1,4,436.516,2.37562,119.8996,119.28,1
75927,22,1,6,5,1,7,2,114,2,1,3,317.370,2.05462,83.2152,71.53,1


In [13]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20)
scaler_x = StandardScaler().fit(x_train)
x_train_scal = scaler_x.transform(x_train)
x_test_scal = scaler_x.transform(x_test)

In [ ]:
gb = HistGradientBoostingClassifier(class_weight='balanced')
pg = {'learning_rate': [0.01, 0.1, 1],
      'max_depth': [None, 1, 3, 5, 7, 10],
      'max_iter': [100, 200, 300],
      'max_leaf_nodes': [30, 40, 50],
      'min_samples_leaf': [10, 20, 30],
      'l2_regularization': [0, 0.3, 0.6]
      }
cv = RepeatedKFold(n_splits=5, n_repeats=5)
gs = GridSearchCV(gb, pg, verbose=3, cv=cv, refit='balanced_accuracy', scoring=('f1', 'balanced_accuracy'))

gs.fit(x_train_scal, y_train)

Fitting 25 folds for each of 1458 candidates, totalling 36450 fits
[CV 1/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.817) f1: (test=0.870) total time=   2.0s
[CV 2/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.797) f1: (test=0.871) total time=   1.0s
[CV 3/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.812) f1: (test=0.873) total time=   1.0s
[CV 4/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.804) f1: (test=0.871) total time=   1.8s
[CV 5/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.813) f1: 

In [ ]:
gs.best_estimator_

In [7]:
y_pred = gs.predict(x_test_scal)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel().tolist()
precision = tp / (tp + fp)
recall = tp / (tp + fn)

In [8]:
print(f'ACC = {accuracy_score(y_test, y_pred):.3f}')
print(f'BA = {balanced_accuracy_score(y_test, y_pred):.3f}')
print(f'MCC = {matthews_corrcoef(y_test, y_pred):.3f}')
print(f'ROC AUC = {roc_auc_score(y_test, y_pred):.3f}')
print(f'F1 = {f1_score(y_test, y_pred):.3f}')
print(f'Precision = {precision:.3f}')
print(f'Recall = {recall:.3f}')

ACC = 0.920
BA = 0.687
MCC = 0.488
ROC AUC = 0.687
F1 = 0.956
Precision = 0.933
Recall = 0.981


In [8]:
print(f'ACC = {accuracy_score(y_test, y_pred):.3f}') # balanced
print(f'BA = {balanced_accuracy_score(y_test, y_pred):.3f}')
print(f'MCC = {matthews_corrcoef(y_test, y_pred):.3f}')
print(f'ROC AUC = {roc_auc_score(y_test, y_pred):.3f}')
print(f'F1 = {f1_score(y_test, y_pred):.3f}')
print(f'Precision = {precision:.3f}')
print(f'Recall = {recall:.3f}')

ACC = 0.824
BA = 0.829
MCC = 0.463
ROC AUC = 0.829
F1 = 0.894
Precision = 0.978
Recall = 0.823
